### Downstream Analysis

Post analysis after zonation trajectory inference:
- Differentially expressed genes / metabolites (Feature assotiation tests)
- Expression imputation via thin-plane spline(TPS) / cubic interpolation?
- Gene-Metabolite cross-correlations
- Gender comparison, etc.

In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, trajectory, configs, zonation
import vgae, model_train, model_eval
import scFates as scf

In [ ]:
from importlib import reload
reload(trajectory)

Load data & latent representations:

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'

sample_id = 'NIH_F5'
adata = IO.load_xenium(os.path.join(xenium_path, sample_id)) 
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata, adata_desi = IO.filter_cells(adata, adata_desi)

# Load latent representations
latent = utils.pnorm(np.load('../results/vMF_8.npy')) 
n_latent = latent.shape[1]

adata.obsm['X_z'] = latent
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)

# Latent representation assignmeng for DESI
adata_desi.obsm['X_z'] = latent
adata_desi.obsm['X_umap'] = adata.obsm['X_umap']

Trajectory inference:

---

In [ ]:
prior_choice = 'vMF'

# Xenium
scf.tl.curve(
    adata,
    use_rep='X_z',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    dist_metric=prior_choice, 
    k=0
    # k=1, n_points=1000
)

# DESI
scf.tl.curve(
    adata_desi,
    use_rep='X_z',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata_desi, 
    use_rep='X_z',
    dist_metric=prior_choice, 
    k=0
    # k=1, n_points=1000
)
                              
gc.collect()

In [ ]:
# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']
adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype(int).max() - adata_desi.obs['milestones'].astype(int)
adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype('category')

In [ ]:
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent)))

sc.pl.umap(
    adata, color='t',
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='coolwarm'
)

sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False
)

#### Test Associations

In [ ]:
scf.tl.test_association(adata, n_jobs=20)
scf.tl.test_association(adata, reapply_filters=True, A_cut=0.1)

In [ ]:
scf.tl.test_association(adata_desi, n_jobs=20)
scf.tl.test_association(adata_desi, reapply_filters=True, A_cut=0.1)

In [ ]:
# Fitting significant features: select PV-end "tip" as "root"

# scf.tl.root(adata, 9) 
# scf.tl.fit(adata, n_jobs=20)

scf.tl.root(adata_desi, 9)
scf.tl.fit(adata_desi, n_jobs=20)

gc.collect()

**Notes**<br>Significant DEGs: directly subsetting adata<br>
Fitted DEG expressions: `adata.layers['fitted']`

In [ ]:
scf.pl.trends(adata, 
              features=adata.var_names,
              highlight_features='fdr',
              ordering='max',
              n_features=10,
              fontsize=7,
              plot_emb=False)

In [ ]:
scf.pl.trends(adata_desi, 
              features=adata_desi.var_names,
              highlight_features='fdr',
              ordering='max',
              n_features=10,
              fontsize=5,
              plot_emb=False)

In [ ]:
# Xenium significant features
scf.pl.single_trend(adata, feature='ADH1C', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)
scf.pl.single_trend(adata, feature='HAMP', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)
scf.pl.single_trend(adata, feature='DPT', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)

gc.collect()

In [ ]:
# DESI significant features
scf.pl.single_trend(adata_desi, feature='FA 22:5', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)
scf.pl.single_trend(adata_desi, feature='PA (38:4)', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)
scf.pl.single_trend(adata_desi, feature='AMP', basis='umap', 
                    alpha_expr=0.1, size_expr=0.01)

gc.collect()

In [ ]:
# FA_markers = [label for label in adata_desi.var_names if 'FA' in label]
# PE_markers = [label for label in adata_desi.var_names if 'PE' in label]
# PI_markers = [label for label in adata_desi.var_names if 'PI' in label]
# PS_markers = [label for label in adata_desi.var_names if 'PS' in label]

# scf.pl.matrix(adata_desi, FA_markers, nbins=100,
#               figsize=(8, len(FA_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PE_markers, nbins=100,
#               figsize=(8, len(PE_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PI_markers, nbins=100,
#               figsize=(8, len(PI_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PS_markers, nbins=100,
#               figsize=(8, len(PS_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')


#### Cell-type dynamics

- (1). Cell type compositions & (2) co-localization dynamics along the trajectory

In [ ]:
# Load cell-type annotation
annots = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
annots = annots.loc[adata.obs_names]


In [ ]:
def get_dynamics(adata, annots, window_size=1000):
    """
    Compute cell-type dynamics along binned trajectory (via sliding window)
    """
    assert 't' in adata.obs.columns, \
        "Please infer zonation trajectory first"

    annots = annots.loc[adata.obs_names]
    annots = annots.loc[adata.obs['t'].sort_values().index]    
    
    cell_types = [cell_type for cell_type in np.unique(annots)
              if cell_type != 'Other' and cell_type != 'Unknown']
    window_size = min(window_size, adata.shape[0]//2)
    n_cell_types = len(cell_types)
    nbins = annots.shape[0] // window_size 
    if annots.shape[0] % window_size != 0:
        nbins += 1
    dynamics = np.zeros((nbins, n_cell_types))  # Column: indiv. cell types
        
    idxl = 0
    for i in range(nbins):
        idxr = annots.shape[0] if i == nbins-1 else idxl+window_size
        summary = annots[idxl:idxr].value_counts()[cell_types]
        dynamics[i] = (summary / summary.sum()).values
        idxl += window_size
    
    return pd.DataFrame(
        dynamics,
        columns=cell_types  
    )


def disp_dynamics(dynamics_df, ncol=4):
    nbins, n_cell_types = dynamics_df.shape
    nrow = n_cell_types // ncol
    if n_cell_types % ncol != 0:
        nrow += 1

    idx = 0
    x = np.arange(nbins)
    f = lambda x, a, b, c, d: a*x**3 + b*x**2 + c*x + d
    
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2), dpi=300)
    for row in range(nrow):
        for col in range(ncol):
            if idx >= n_cell_types:
                axes[row, col].axis('off')
                continue

            y = dynamics_df.iloc[:, idx]
            xx = np.linspace(x.min(), x.max(), 500)
            a, b, c, d = np.polyfit(x, y, 3)
            yy = f(xx, a, b, c, d)
            
            axes[row, col].scatter(x, dynamics_df.iloc[:, idx], 
                               s=1, c='k', alpha=0.5)
            axes[row, col].plot(xx, yy, color='b', linewidth=2, alpha=0.2)
            axes[row, col].set_title(dynamics_df.columns[idx], fontsize=12)
            axes[row, col].set_xlabel('PV -> CV\n (sliding windows)', fontsize=10)
            axes[row, col].set_ylabel('Proportions')
            axes[row, col].spines[['right', 'top']].set_visible(False)
            idx += 1
    
    
    plt.tight_layout()
    plt.show()


In [ ]:
dynamics_df = get_dynamics(adata, annots, window_size=500)
disp_dynamics(dynamics_df, ncol=4)

#### Xenium - DESI correlations

In [ ]:
fitted_xenium = adata.layers['fitted']  # [N x G]
fitted_desi = adata_desi.layers['fitted']  # [N x M]
fitted_concat = np.hstack((fitted_xenium, fitted_desi))
fitted_concat.shape

In [ ]:
corr = np.corrcoef(fitted_concat.T)
corr_df = pd.DataFrame(corr[:adata.shape[1], adata.shape[1]:], 
                       index=list(adata.var_names),
                       columns=list(adata_desi.var_names))

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
g.ax_heatmap.set_xlabel

In [ ]:
g = sns.clustermap(
    corr_df,
    method='ward',
    cmap='RdBu_r',
)

g.ax_heatmap.set_xlabel('Metabolites', fontsize=15)
g.ax_heatmap.set_ylabel('Genes', fontsize=15)


In [ ]:
### Extract the clustered data
g = sns.clustermap(corr_df, cmap='RdBu_r')
clustered_data = g.data2d

### Create a dendrogram for the rows
dendro_row = ff.create_dendrogram(clustered_data.values, orientation='left', labels=clustered_data.index)
for trace in dendro_row['data']:
    trace['yaxis'] = 'y2'  # Linking this dendrogram to a different y-axis

### Create a dendrogram for the columns
dendro_col = ff.create_dendrogram(clustered_data.values.T, orientation='bottom', labels=clustered_data.columns)
for trace in dendro_col['data']:
    trace['xaxis'] = 'x2'  # Linking this dendrogram to a different x-axis

### Create main heatmap
heatmap = go.Heatmap(
    z=clustered_data.values,
    x=clustered_data.columns,
    y=clustered_data.index,
    colorscale='RdBu_r'
)

In [ ]:
### Combine Dendrograms and Heatmap into a Single Plotly Figure
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.2, 0.8], column_widths=[0.8, 0.2],
    specs=[[{"type": "xy"}, None],
           [{"type": "heatmap"}, {"type": "xy"}]],
    horizontal_spacing=1, vertical_spacing=0.1
)

# Add the column dendrogram (top side)
for trace in dendro_col['data']:
    fig.add_trace(trace, row=1, col=1)

# Add the row dendrogram (right side)
for trace in dendro_row['data']:
    fig.add_trace(trace, row=2, col=2)

# Add the heatmap (bottom-left corner)
fig.add_trace(heatmap, row=2, col=1)

### Layout
# Update specific axes settings
fig.update_xaxes(domain=[0, 0.8], row=2, col=1)  # Heatmap X-axis
fig.update_yaxes(domain=[0.04, 0.76], row=2, col=1)  # Heatmap Y-axis

# Column dendrogram
fig.update_xaxes(domain=[0, 0.8], showticklabels=False, row=1, col=1)  
fig.update_yaxes(domain=[0.78, 1], showticklabels=False, row=1, col=1)  

# Row dendrogram
fig.update_xaxes(domain=[0.82, 1], showticklabels=False, row=2, col=2) 
fig.update_yaxes(domain=[0, 0.8], showticklabels=False, row=2, col=2) 

### Final layout adjustments
fig.update_layout(
    height=800, width=800,
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='white',
    margin=dict(l=100, r=50, t=50, b=100),
)



In [ ]:
fig.write_html("../sketch/Xenium_DESI_heatmap.html")
gc.collect()

---